2. Superstore Profit Leakage Investigation

A regional sales head says total sales look healthy, but profit is not improving. You are given superstore order data with customer, segment, region, category, sub-category, sales, quantity, discount, profit, order date, and ship date. Investigate whether the business has a discount-driven profit leakage problem. Identify orders where high sales still resulted in negative profit, categories or sub-categories where discounts are repeatedly connected with losses, suspicious rows where profit or sales do not match the business expectation, and regions where the issue is more visible. Your final response should separate high-revenue business from genuinely profitable business and explain which records should be reviewed by the pricing or sales team.

In [1]:
import pandas as pd

df = pd.read_csv(r"D:\vs code\Superstore-problem01\Sample - Superstore.csv", encoding='latin1')

print(df.shape)
df.head()

(9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [4]:
df.isnull().sum()
print("Number of missing values in each column:", df.isnull().sum())

Number of missing values in each column: Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [5]:
df.duplicated().sum()
print("Number of duplicate rows:", df.duplicated().sum())


Number of duplicate rows: 0


In [6]:
df["Order Date"]=pd.to_datetime(df["Order Date"])
df["Ship Date"]=pd.to_datetime(df["Ship Date"])

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   str           
 2   Order Date     9994 non-null   datetime64[us]
 3   Ship Date      9994 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   Country        9994 non-null   str           
 9   City           9994 non-null   str           
 10  State          9994 non-null   str           
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   str           
 13  Product ID     9994 non-null   str           
 14  Category       9994 non-null   str           
 15  Sub-Category   9994 non-null   s

In [8]:
print("First Order Date :", df["Order Date"].min())
print("Last Order Date :", df["Order Date"].max())

df["Year"] = df["Order Date"].dt.year
df["Year"].value_counts().sort_index()

First Order Date : 2014-01-03 00:00:00
Last Order Date : 2017-12-30 00:00:00


Year
2014    1993
2015    2102
2016    2587
2017    3312
Name: count, dtype: int64

In [9]:
#Standarize text columns
df.select_dtypes(include=["object", "string"]).columns
text_cols = df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    df[col] = df[col].str.strip()

In [10]:
df["Segment"] = df["Segment"].str.title()
df["Region"] = df["Region"].str.title()
df["Category"] = df["Category"].str.title()
df["Sub-Category"] = df["Sub-Category"].str.title()

In [11]:
df["Segment"].value_counts()
df["Region"].value_counts()
df["Category"].value_counts()
df["Sub-Category"].value_counts()

for col in ["Segment","Region","Category","Sub-Category"]:
    print(col)
    print(df[col].unique())
    print()


Segment
<StringArray>
['Consumer', 'Corporate', 'Home Office']
Length: 3, dtype: str

Region
<StringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str

Category
<StringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str

Sub-Category
<StringArray>
[  'Bookcases',      'Chairs',      'Labels',      'Tables',     'Storage',
 'Furnishings',         'Art',      'Phones',     'Binders',  'Appliances',
       'Paper', 'Accessories',   'Envelopes',   'Fasteners',    'Supplies',
    'Machines',     'Copiers']
Length: 17, dtype: str



In [12]:
df[["Sales","Profit","Discount","Quantity"]].describe().T


,count,mean,std,min,25%,50%,75%,max
Sales,9994.0,229.858001,623.245101,0.444,17.28000,54.4900,209.940,22638.480
Profit,9994.0,28.656896,234.260108,-6599.978,1.72875,8.6665,29.364,8399.976
Discount,9994.0,0.156203,0.206452,0.000,0.00000,0.2000,0.200,0.800
Quantity,9994.0,3.789574,2.225110,1.000,2.00000,3.0000,5.000,14.000


Columns Metrics

In [13]:
#Shipping coulmn 
df["shipping_days"] = (df["Ship Date"] -df["Order Date"]).dt.days
print("Average shipping days:", df["shipping_days"].mean())
print("Minimum shipping days:", df["shipping_days"].min())
print("Maximum shipping days:", df["shipping_days"].max())
print("Shipping days distribution:")
print(df["shipping_days"].value_counts().sort_index())

Average shipping days: 3.958174904942966
Minimum shipping days: 0
Maximum shipping days: 7
Shipping days distribution:
shipping_days
0     519
1     369
2    1334
3    1005
4    2774
5    2169
6    1203
7     621
Name: count, dtype: int64


In [14]:
df[df["Sales"] == 0]
print("Number of rows with zero sales:", df[df["Sales"] == 0].shape[0])

Number of rows with zero sales: 0


In [15]:
#Profit margin column
df["profit_margin"] = (df["Profit"] /df["Sales"]) * 100
print("Average profit margin:", df["profit_margin"].mean())
print("Minimum profit margin:", df["profit_margin"].min())  
print("Maximum profit margin:", df["profit_margin"].max())
print("Profit margin distribution:")
pd.cut(df["profit_margin"],bins=[-1000,0,10,20,30,1000]).value_counts().sort_index(ascending=False)

Average profit margin: 12.031392972104458
Minimum profit margin: -275.0
Maximum profit margin: 50.0
Profit margin distribution:


profit_margin
(30, 1000]    4145
(20, 30]      1753
(10, 20]       934
(0, 10]       1226
(-1000, 0]    1936
Name: count, dtype: int64

In [16]:
#Loss Order Flag
df["loss_order"] = df["Profit"] < 0
print("Number of loss orders:", df["loss_order"].sum())
print("Percentage of loss orders:", (df["loss_order"].mean() * 100).round(2), "%")

Number of loss orders: 1871
Percentage of loss orders: 18.72 %


In [17]:
#High Discount Flag
df["high_discount"] = df["Discount"] >= 0.20
print("Number of high discount orders:", df["high_discount"].sum())
print("Percentage of high discount orders:", (df["high_discount"].mean() * 100).round(2), "%")

Number of high discount orders: 5050
Percentage of high discount orders: 50.53 %


In [18]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Product Name,Sales,Quantity,Discount,Profit,Year,shipping_days,profit_margin,loss_order,high_discount
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,3,16.00,False,False
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,3,30.00,False,False
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,4,47.00,False,False
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,7,-40.00,True,True
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,7,11.25,False,True


Investigation 

1. Which weekday generates the lowest total sales and profit?


In [107]:
df['weekday'] = df['Order Date'].dt.day_name()

weekday_summary = (df.groupby('weekday') .agg(Sales=('Sales','sum'),Profit=('Profit','sum'),Orders=('Order ID','nunique')).reset_index())
weekday_summary['Sales_Per_Order'] = (weekday_summary['Sales'] /weekday_summary['Orders'])
weekday_summary['Profit_Per_Order'] = ( weekday_summary['Profit'] /weekday_summary['Orders'])

weekday_summary.sort_values('Sales', ascending=False)
print("Sales and profit by weekday:")
print(weekday_summary.sort_values('Sales', ascending=False))

Sales and profit by weekday:
     weekday        Sales      Profit  Orders  Sales_Per_Order  \
1     Monday  428937.8157  51511.1146     920       466.236756   
0     Friday  427692.2593  46520.8242     916       466.912947   
3     Sunday  385353.8464  57129.5120     850       453.357466   
2   Saturday  357984.6830  41277.6202     837       427.699741   
4   Thursday  321965.7785  39683.7512     746       431.589515   
5    Tuesday  287209.5022  35570.3288     558       514.712370   
6  Wednesday   88056.9752  14703.8707     182       483.829534   

   Profit_Per_Order  
1         55.990342  
0         50.786926  
3         67.211191  
2         49.316153  
4         53.195377  
5         63.746109  
6         80.790498  


In [102]:
weekday_summary['Sales'].idxmin()
print("Worst weekday for sales:", weekday_summary['Sales'].idxmin())

weekday_summary['Profit'].idxmin()
print("Worst weekday for profit:", weekday_summary['Profit'].idxmin())


Worst weekday for sales: Wednesday
Worst weekday for profit: Wednesday


BUSINESS INSIGHT : Wednesday recorded the lowest sales ($88K) and profit ($14.7K) among all weekdays, indicating weaker business performance compared to the rest of the week.

2. Which month has the lowest sales and profit?

In [ ]:
df['month_num'] = df['Order Date'].dt.month
df['month'] = df['Order Date'].dt.month_name()

month_summary = (df.groupby(['month_num', 'month']) .agg(Sales=('Sales', 'sum'),Profit=('Profit', 'sum')).sort_values('Sales'))
print("Sales and profit by month:")
print(month_summary.sort_values('Sales', ascending=False))

month_summary

month_summary
# Lowest sales month
month_summary['Sales'].idxmin()
print("Worst month for sales:", month_summary['Sales'].idxmin())

# Lowest profit month
month_summary['Profit'].idxmin()
print("Worst month for profit:", month_summary['Profit'].idxmin())

Sales and profit by month:
                           Sales      Profit
month_num month                             
11        November   352461.0710  35468.4265
12        December   325293.5035  43369.1919
9         September  307649.9457  36857.4753
3         March      205005.4888  28594.6872
10        October    200322.9847  31784.0413
8         August     159044.0630  21776.9384
5         May        155028.8117  22411.3078
6         June       152718.6793  21285.7954
7         July       147238.0970  13832.6648
4         April      137762.1286  11587.4363
1         January     94924.8356   9134.4461
2         February    59751.2514  10294.6107
Worst month for sales: (np.int32(2), 'February')
Worst month for profit: (np.int32(1), 'January')


BUSINESS INSIGHT : February struggled with revenue generation, while January struggled more with profitability. This indicates that low sales and low profit are not always driven by the same factors. Sales were lowest in February ($59.8K), while profit was lowest in January ($9.1K). This suggests weaker business performance during the beginning of the year, whereas November was the best-performing month in terms of sales.

3. discounts actually causing the losses

In [87]:
df['high_discount'] = df['Discount'] > 0.20
discount_impact = (df.groupby('high_discount').agg( Avg_Sales=('Sales', 'mean'), Avg_Profit=('Profit', 'mean') ))

discount_impact


,Avg_Sales,Avg_Profit
high_discount,,
False,224.907651,49.037679
True,260.423654,-97.183098


BUSINESS INSIGHT : Customers who receive larger discounts are not spending much more than other customers, but these orders are generating significantly lower profits. This suggests that heavy discounting is one of the main reasons why sales remain strong while overall profitability suffers.

4. At what discount percentage does average profit become negative?

In [20]:
df["discount_bucket"] = pd.cut(df["Discount"],bins=[-0.01,0,0.1,0.2,0.3,0.5,0.8],labels=["0%","0-10%","10-20%","20-30%","30-50%","50-80%"])

df.groupby("discount_bucket").agg({ "Sales":"mean", "Profit":"mean"})

,Sales,Profit
discount_bucket,,
0%,226.742074,66.900292
0-10%,578.397351,96.055074
10-20%,213.575867,24.738824
20-30%,454.742974,-45.679636
30-50%,630.047603,-156.282991
50-80%,75.033572,-89.438144


 BUSINESS INSIGHT: As the discount percentage increases, average profit starts to fall and eventually turns negative. This suggests that offering larger discounts is reducing profitability and may be one of the main reasons for profit leakage.

5. High-Sales, Negative-Profit Orders

In [23]:
df["Sales"].median()
print("Median Sales:", df["Sales"].median())

Median Sales: 54.489999999999995


In [ ]:
high_sales_threshold = df["Sales"].median()
high_sales_loss = df[(df["Sales"] > high_sales_threshold) &(df["Profit"] < 0)]

high_sales_loss.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sales,Quantity,Discount,Profit,Year,shipping_days,profit_margin,loss_order,high_discount,discount_bucket
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,957.5775,5,0.45,-383.0310,2015,7,-40.000000,True,True,30-50%
14,15,US-2015-118983,2015-11-22,2015-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,68.8100,5,0.80,-123.8580,2015,4,-180.000000,True,True,50-80%
23,24,US-2017-156909,2017-07-16,2017-07-18,Second Class,SF-20065,Sandra Flanagan,Consumer,United States,Philadelphia,...,71.3720,2,0.30,-1.0196,2017,2,-1.428571,True,True,20-30%
27,28,US-2015-150630,2015-09-17,2015-09-21,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,...,3083.4300,7,0.50,-1665.0522,2015,4,-54.000000,True,True,30-50%
36,37,CA-2016-117590,2016-12-08,2016-12-10,First Class,GH-14485,Gene Hale,Corporate,United States,Richardson,...,190.9200,5,0.60,-147.9630,2016,2,-77.500000,True,True,50-80%


BUSINESS INSIGHT : Several orders generated high sales but still resulted in losses. This indicates that strong revenue does not always translate into profitability, with heavy discounts likely eroding margins and turning large orders into loss-making transactions.

Product Analysis


6.Which products are causing the losses?

In [56]:
df.groupby("Category").agg({"Sales":"sum","Profit":"sum", "Discount":"mean"})


,Sales,Profit,Discount
Category,,,
Furniture,741999.7953,18451.2728,0.173923
Office Supplies,719047.0320,122490.8008,0.157285
Technology,836154.0330,145454.9481,0.132323


In [57]:
df.groupby("Sub-Category").agg({ "Sales":"sum", "Profit":"sum", "Discount":"mean"}).sort_values("Profit")

,Sales,Profit,Discount
Sub-Category,,,
Tables,206965.5320,-17725.4811,0.261285
Bookcases,114879.9963,-3472.5560,0.211140
Supplies,46673.5380,-1189.0995,0.076842
Fasteners,3024.2800,949.5182,0.082028
Machines,189238.6310,3384.7569,0.306087
Labels,12486.3120,5546.2540,0.068681
Art,27118.7920,6527.7870,0.074874
Envelopes,16476.4020,6964.1767,0.080315
Furnishings,91705.1640,13059.1436,0.138349


BUSINESS INSIGHT: Tables and Bookcases are the biggest loss-making products. Despite generating strong sales, high discount levels appear to be reducing profitability and contributing to overall business losses.

Discounting is likely a major factor behind losses in Tables and Bookcases, while losses in Supplies may be driven by different factors such as pricing, costs, or product mix.

7. Which customer type benefits from excessive discounting?

In [ ]:
df['high_discount'] = df['Discount'] > 0.20
df.groupby('Segment').agg( Avg_Discount=('Discount','mean'),Avg_Sales=('Sales','mean'),Avg_Profit=('Profit','mean'))
df.groupby(['Segment','high_discount']).agg(Orders=('Order ID','nunique'),Avg_Sales=('Sales','mean'),Avg_Profit=('Profit','mean'))

Orders   Avg_Sales  Avg_Profit
Segment     high_discount                                
Consumer    False            2361  218.172947   46.231905
            True              534  257.445844  -97.809743
Corporate   False            1389  233.374417   50.673037
            True              309  236.602101  -94.692293
Home Office False             835  230.083221   54.377871
            True              177  311.657868  -99.643412

BUSINESS INSIGHT : Customers who received discounts above 20% spent more per order, but those orders resulted in losses across all customer segments. This suggests that heavy discounting increases sales but hurts profitability.

8. Which products contribute the largest share of total losses?

In [31]:
loss_products = (
    df[df["Profit"] < 0]
      .groupby("Sub-Category")["Profit"]
      .sum()
      .sort_values()
)
[print(f"{sub_cat}: ${loss:.2f}") for sub_cat, loss in loss_products.items()]

Binders: $-38510.50
Tables: $-32412.15
Machines: $-30118.67
Bookcases: $-12152.21
Chairs: $-9880.84
Appliances: $-8629.64
Phones: $-7530.62
Furnishings: $-6490.91
Storage: $-6426.30
Supplies: $-3015.62
Accessories: $-930.63
Fasteners: $-33.20


[None, None, None, None, None, None, None, None, None, None, None, None]

BUSINESS INSIGHT: Binders were the biggest source of losses ($38.5K), followed by Tables ($32.4K) and Machines ($30.1K). These three sub-categories account for a substantial portion of total loss-making orders and should be prioritized for pricing and discount review.

9. Which products need discounts to sell?

In [120]:
df.groupby(['Sub-Category','high_discount']).agg( Orders=('Order ID','nunique'), 
            Sales=('Sales','sum'),Profit=('Profit','sum')
)

Orders        Sales      Profit
Sub-Category high_discount                                 
Accessories  False             718  167380.3180  41936.6357
Appliances   False             386  104149.6270  26767.6466
             True               65    3382.5340  -8629.6412
Art          False             731   27118.7920   6527.7870
Binders      False             790  167272.1200  68732.2597
             True              526   36140.6130 -38510.4964
Bookcases    False             155   86335.9975   7625.2054
             True               69   28543.9988 -11097.7614
Chairs       False             427  258443.6010  33327.2830
             True              149   70005.5020  -6737.1167
Copiers      False              59  132608.2280  53434.8497
             True                9   16919.8020   2182.9752
Envelopes    False             249   16476.4020   6964.1767
Fasteners    False             215    3024.2800    949.5182
Furnishings  False             750   85060.4640  19003.7988
             True              127    6644.7000  -5944.6552
Labels       False             346   12486.3120   5546.2540
Machines     False              62  112399.5230  32940.1090
             True               50   76839.1080 -29555.3521
Paper        False            1191   78479.2060  34053.5693
Phones       False             716  295669.7020  50901.5160
             True               98   34337.3520  -6385.7854
Storage      False             777  223843.6080  21278.8264
Supplies     False             187   46673.5380  -1189.0995
Tables       False             139  117008.9920  12972.7417
             True              168   89956.5400 -30698.2228

BUSINESS INSIGHT: Customers are buying more Tables, Machines, and Binders when large discounts are offered, but the business ends up losing money on many of those sales. These products seem highly dependent on promotions, making them key areas for pricing review.

10. Which product repeatedly creates loss-making orders?

In [39]:
(
    df[df["Profit"] < 0].groupby("Sub-Category").size().sort_values(ascending=False)
)

Sub-Category
Binders        613
Chairs         235
Tables         203
Furnishings    167
Storage        161
Phones         136
Bookcases      109
Accessories     91
Appliances      67
Machines        44
Supplies        33
Fasteners       12
dtype: int64

BUSINESS INSIGHT: Binders consistently appear across multiple loss analyses, generating the highest number of loss-making orders and the largest total losses. This suggests that discounting and pricing decisions for Binders may be a key driver of overall profitability issues.

REGION ANALYSIS 

11. Where is the problem most visible?

In [121]:
df.groupby("Region").agg( Avg_Sales=("Sales","mean"),
    Avg_Profit=("Profit","mean"),
    Avg_Discount=("Discount","mean")
)

,Avg_Sales,Avg_Profit,Avg_Discount
Region,,,
Central,215.772661,17.092709,0.240353
East,238.336110,32.135808,0.145365
South,241.803645,28.857673,0.147253
West,226.493233,33.849032,0.109335


BUSINESS INSIGHT: The Central region appears to be the most problematic area. It has the highest average discount (24%), but the lowest average sales ($216) and lowest average profit ($17) per order. This suggests that heavy discounting in the region is not translating into stronger sales or profitabilit

12. Which region is responsible for the largest dollar loss?

In [122]:
region_loss = (df[df["Profit"] < 0].groupby("Region").agg(
        Loss_Amount=("Profit", "sum"),
        Loss_Orders=("Order ID", "count")).sort_values("Loss_Amount"))

region_loss

,Loss_Amount,Loss_Orders
Region,,
Central,-56314.8850,741
East,-49590.6075,553
South,-27504.8323,259
West,-22720.9609,318


BUSINESS INSIGHT : The Central region is the biggest contributor to profit leakage, with over $56K in losses across 741 loss-making orders. This indicates that the profitability issue is most severe in the Central region and should be investigated first.

13. Dangerous Products Within Regions

In [50]:
region_product = (df.groupby(["Region","Sub-Category"]).agg(Sales=("Sales","sum"),Profit=("Profit","sum"))
      .reset_index())
region_product[region_product["Profit"] < 0].sort_values("Profit")

,Region,Sub-Category,Sales,Profit
33,East,Tables,39139.8070,-11025.3801
50,South,Tables,43916.1920,-4623.0579
9,Central,Furnishings,15254.3700,-3906.2168
16,Central,Tables,39154.9710,-3559.6504
1,Central,Appliances,23582.0330,-2638.6175
4,Central,Bookcases,24157.1768,-1997.9043
55,West,Bookcases,36004.1235,-1646.5117
11,Central,Machines,26797.3840,-1486.0666
45,South,Machines,53890.9600,-1438.8930
21,East,Bookcases,43819.3340,-1167.6318


BUSINESS INSIGHT: Tables are consistently losing money across regions, while the Central region shows losses across multiple product categories. This suggests that both product pricing and discount strategies need closer review in the Central region.

SHIPPING ANALYSIS 

14. Are we spending more on faster shipping for orders that still lose money?

In [123]:
df.groupby(["Ship Mode", "loss_order"]).agg(Orders=("Order ID", "count"),Sales=("Sales", "sum"),
    Profit=("Profit", "sum"))

print("Loss orders by shipping mode:")
print(df.groupby(["Ship Mode", "loss_order"]).agg(Orders=("Order ID", "count"),Sales=("Sales", "sum"),
    Profit=("Profit", "sum")))

Loss orders by shipping mode:
                           Orders         Sales       Profit
Ship Mode      loss_order                                   
First Class    False         1245  2.898683e+05   71630.1108
               True           293  6.156010e+04  -22660.2709
Same Day       False          445  9.990228e+04   24789.8643
               True            98  2.846084e+04   -8898.1054
Second Class   False         1638  3.780010e+05   81671.0083
               True           307  8.119252e+04  -24224.3729
Standard Class False         4795  1.060722e+06  264437.3240
               True          1173  2.974937e+05 -100348.5365


15. products are most frequently shipped using premium shipping but remain unprofitable?

In [ ]:
df.groupby(["Ship Mode", "Sub-Category"]).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
).query("Profit < 0")

print("Loss orders by shipping mode and sub-category:")
print(df.groupby(["Ship Mode", "Sub-Category"]).agg(Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
).query("Profit < 0"))

Loss orders by shipping mode and sub-category:
                                   Sales      Profit
Ship Mode      Sub-Category                         
First Class    Bookcases      21065.3189   -305.9975
               Machines       24153.4010  -1603.4990
               Tables         28800.7760  -1365.3665
Same Day       Tables          9644.3470  -1129.4225
Second Class   Bookcases      22307.2129  -1028.0213
               Supplies       29378.4100   -505.5624
               Tables         43693.7475  -3320.6799
Standard Class Bookcases      68536.8825  -2531.1028
               Supplies       13216.5240   -875.5835
               Tables        124826.6615 -11910.0122


BUSINESS INSIGHT: Tables continue to lose money across every shipping mode, making them the biggest contributor to profit leakage. Strong sales are not translating into profits, indicating a pricing or discounting issue.

The company has a profit leakage problem driven mainly by excessive discounting, loss-making product categories, and poor profitability in the Central region.


1. Heavy discounts are hurting profits.
2. High sales don't always mean high profit
3. Tables, Bookcases, Binders, and Machines are the main loss drivers.
4. Central region needs immediate attention.